# Notebook 3 — Simple ETL Pipeline

**Use Case 2 goal:** demonstrate the ETL pattern end to end — **Extract → Transform → Aggregate → Load**.

## What learners should understand
- how the cleaned file becomes an ETL input
- how aggregation turns row-level transactions into reporting-ready outputs
- how the same business logic can later be recreated in AWS Glue with PySpark


## Dependencies, AWS setup, and files used

### Python packages
- **boto3** — AWS-friendly access to the cleaned file in S3 and the final ETL outputs written back to S3
- **pandas** — easiest way to explain ETL aggregation logic cell by cell
- **io** — bridges S3 object content and pandas DataFrames
- **pathlib** — fallback for offline rehearsal


In [1]:
from pathlib import Path
import pandas as pd

# ✅ Pure local execution setup (no S3 calls, no boto3 credentials required)
LOCAL_INPUT_PATH = Path('./retail_cleaned.csv')
LOCAL_DAILY_PATH = Path('./daily_country_revenue.csv')
LOCAL_MONTHLY_PATH = Path('./monthly_category_revenue.csv')


def read_csv_aws_first(s3_uri: str, local_path: Path) -> pd.DataFrame:
    """Read directly from local file storage."""
    print("📂 Reading from local file:", local_path)
    return pd.read_csv(local_path)


def write_csv_aws_first(df: pd.DataFrame, s3_uri: str, local_path: Path) -> None:
    """Write directly to local file storage."""
    print("📂 Writing locally to:", local_path)
    local_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(local_path, index=False)


print('ETL input:', LOCAL_INPUT_PATH)
print('Daily output:', LOCAL_DAILY_PATH)
print('Monthly output:', LOCAL_MONTHLY_PATH)

ETL input: retail_cleaned.csv
Daily output: daily_country_revenue.csv
Monthly output: monthly_category_revenue.csv


## Step 1 — Extract the cleaned data

### Why this step is performed
In ETL terms, this is the **extract** phase: we load the cleaned source that is ready for downstream calculations and aggregation.


In [3]:
INPUT_S3_URI = "./retail_cleaned.csv"
LOCAL_INPUT_PATH = Path('./retail_cleaned.csv')

df = read_csv_aws_first(INPUT_S3_URI, LOCAL_INPUT_PATH)
print('Clean input shape:', df.shape)
display(df.head())

📂 Reading from local file: retail_cleaned.csv
Clean input shape: (489, 14)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,InvoiceDateParsed,TransactionDate,Year,Month,Revenue,IsReturn
0,536365,71053,WHITE METAL LANTERN,6,2011-02-01 11:08:00,5.49,17889,Belgium,2011-02-01 11:08:00,2011-02-01,2011,2011-02,32.94,False
1,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,2,2011-01-28 11:32:00,4.22,16943,Germany,2011-01-28 11:32:00,2011-01-28,2011,2011-01,8.44,False
2,536366,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2011-02-05 08:48:00,5.80,18065,Netherlands,2011-02-05 08:48:00,2011-02-05,2011,2011-02,34.80,False
3,536366,22752,SET 7 BABUSHKA NESTING BOXES,4,2011-01-13 13:54:00,7.55,14512,United Kingdom,2011-01-13 13:54:00,2011-01-13,2011,2011-01,30.20,False
4,536366,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2011-03-22 17:56:00,4.03,17075,Germany,2011-03-22 17:56:00,2011-03-22,2011,2011-03,24.18,False


## Step 2 — Transform for reporting

### Why this step is performed
Create fields that make aggregation easier.


In [4]:
df['TransactionDate'] = pd.to_datetime(df['TransactionDate'])
df['Category'] = df['Description'].fillna('UNKNOWN_ITEM').str.split().str[0]
df['Revenue'] = df['Quantity'] * df['UnitPrice']

display(df[['TransactionDate', 'Country', 'Category', 'Revenue']].head())


,TransactionDate,Country,Category,Revenue
0,2011-02-01,Belgium,WHITE,32.94
1,2011-01-28,Germany,GLASS,8.44
2,2011-02-05,Netherlands,GLASS,34.80
3,2011-01-13,United Kingdom,SET,30.20
4,2011-03-22,Germany,GLASS,24.18


## Step 3 — Aggregate into reporting outputs

### Why this step is performed
This is where row-level data becomes business-ready output.


In [5]:
daily_country_revenue = (
    df.groupby([df['TransactionDate'].dt.date.astype(str), 'Country'], as_index=False)['Revenue']
      .sum()
      .rename(columns={'TransactionDate': 'Date'})
)

monthly_category_revenue = (
    df.groupby(['Month', 'Category'], as_index=False)['Revenue']
      .sum()
)

display(daily_country_revenue.head())
display(monthly_category_revenue.head())


C:\Users\Admin\AppData\Local\Temp\ipykernel_28136\2479616822.py:3: FutureWarning: A grouping was used that is not in the columns of the DataFrame and so was excluded from the result. This grouping will be included in a future version of pandas. Add the grouping as a column of the DataFrame to silence this warning.
  .sum()


,Country,Revenue
0,France,7.74
1,United Kingdom,63.96
2,Belgium,20.75
3,Germany,50.78
4,Belgium,11.62


,Month,Category,Revenue
0,2011-01,ASSORTED,333.42
1,2011-01,CREAM,314.82
2,2011-01,GLASS,432.14
3,2011-01,HAND,1161.19
4,2011-01,KNITTED,558.09


## Step 4 — Load the ETL outputs back to S3

### Why this step is performed
The daily and monthly views are written back to S3 so they can be verified in the AWS console.


In [7]:
OUT_DAILY_S3_URI = "./daily_country_revenue.csv"
OUT_MONTHLY_S3_URI = "./monthly_category_revenue.csv"
LOCAL_DAILY_PATH = Path('./daily_country_revenue.csv')
LOCAL_MONTHLY_PATH = Path('./monthly_category_revenue.csv')

write_csv_aws_first(daily_country_revenue, OUT_DAILY_S3_URI, LOCAL_DAILY_PATH)
write_csv_aws_first(monthly_category_revenue, OUT_MONTHLY_S3_URI, LOCAL_MONTHLY_PATH)

print('Daily revenue output saved to:')
print(LOCAL_DAILY_PATH.resolve())
print('Monthly revenue output saved to:')
print(LOCAL_MONTHLY_PATH.resolve())

📂 Writing locally to: daily_country_revenue.csv
📂 Writing locally to: monthly_category_revenue.csv
Daily revenue output saved to:
C:\Users\Admin\Desktop\sreedhar\git4_\BITS_programming\module_2\week_6\use_case_2_local\daily_country_revenue.csv
Monthly revenue output saved to:
C:\Users\Admin\Desktop\sreedhar\git4_\BITS_programming\module_2\week_6\use_case_2_local\monthly_category_revenue.csv


In [8]:
import datetime, pytz; 
print("Current Time in IST:", datetime.datetime.now(pytz.utc).astimezone(pytz.timezone('Asia/Kolkata')).strftime('%Y-%m-%d %H:%M:%S'))

Current Time in IST: 2026-08-31 21:53:34
